# TRACER — Module 4: Adversarial Attack Generation

Runs real FGSM and PGD attacks against real CLIP, using real Flickr8k images from Module 3.
Self-contained — clones the repo and re-downloads Flickr8k from Drive automatically.

In [ ]:
GITHUB_USERNAME = "IqraYasmin123"   # change if this isn't your username
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/tracers.git"

import os
if not os.path.exists("/content/tracers"):
    !git clone {REPO_URL} /content/tracers
else:
    %cd /content/tracers
    !git pull

%cd /content/tracers/ai-engine
import sys
sys.path.insert(0, ".")

from src.vlm import CLIPVLM, VLMConfig
from src.dataset import FlickrDataset, DatasetConfig
from src.attacks import FGSMAttack, PGDAttack, AttackConfig
import torch
print("Imports OK.")


## Load CLIP and the Flickr8k dataset (from Drive, downloaded in Module 3)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

vlm = CLIPVLM(VLMConfig())
print(f"CLIP loaded on {vlm.device}")

DATASET_DIR = "/content/drive/MyDrive/TRACER/datasets/flickr8k"
dataset_config = DatasetConfig(root_dir=DATASET_DIR, image_size=(224, 224))
dataset = FlickrDataset(dataset_config)
print(f"Loaded {len(dataset)} (image, caption) pairs.")


## Pick a real sample and attack it

In [ ]:
sample = dataset[0]
real_image = sample["image"]
true_caption = sample["caption"]
print(f"Caption: {true_caption}")
real_image


In [ ]:
pixel_values = vlm.preprocess_image(real_image)
text_embeds = vlm.encode_text([true_caption])

clean_sim = vlm.similarity(vlm.encode_image(pixel_values), text_embeds).item()
print(f"Clean similarity to true caption: {clean_sim:.4f}")


## Run FGSM

`epsilon = 8/255` is a standard, commonly-used budget in adversarial ML literature — small
enough to be visually near-imperceptible, large enough to meaningfully fool most models.

In [ ]:
fgsm_config = AttackConfig(epsilon=8/255)
fgsm_adv = FGSMAttack(fgsm_config).generate(vlm, pixel_values, text_embeds)

fgsm_sim = vlm.similarity(vlm.encode_image(fgsm_adv), text_embeds).item()
print(f"Clean similarity:      {clean_sim:.4f}")
print(f"FGSM adversarial sim:  {fgsm_sim:.4f}  (change: {fgsm_sim - clean_sim:+.4f})")


## Run PGD (stronger — should degrade similarity more than FGSM)

In [ ]:
pgd_config = AttackConfig(epsilon=8/255, alpha=2/255, steps=10)
pgd_adv = PGDAttack(pgd_config).generate(vlm, pixel_values, text_embeds)

pgd_sim = vlm.similarity(vlm.encode_image(pgd_adv), text_embeds).item()
print(f"Clean similarity:     {clean_sim:.4f}")
print(f"FGSM adversarial sim: {fgsm_sim:.4f}  (change: {fgsm_sim - clean_sim:+.4f})")
print(f"PGD adversarial sim:  {pgd_sim:.4f}  (change: {pgd_sim - clean_sim:+.4f})")


## Visual comparison: clean vs. FGSM vs. PGD

The three images should look nearly identical to the human eye — that's the point of the
epsilon bound. The similarity scores above, not your eyes, are what reveal the attack.

In [ ]:
import matplotlib.pyplot as plt

def to_display(pixel_values, processor):
    mean = torch.tensor(processor.image_processor.image_mean).view(1, 3, 1, 1).to(vlm.device)
    std = torch.tensor(processor.image_processor.image_std).view(1, 3, 1, 1).to(vlm.device)
    img = (pixel_values * std + mean).clamp(0, 1)
    return img.squeeze().permute(1, 2, 0).detach().cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(to_display(pixel_values, vlm.processor))
ax[0].set_title(f"Clean\nsim={clean_sim:.3f}")
ax[1].imshow(to_display(fgsm_adv, vlm.processor))
ax[1].set_title(f"FGSM\nsim={fgsm_sim:.3f}")
ax[2].imshow(to_display(pgd_adv, vlm.processor))
ax[2].set_title(f"PGD\nsim={pgd_sim:.3f}")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()


## Module 4 completion check

Run the full test suite:
```bash
cd ai-engine
pytest tests/test_attacks.py -v
```
All 11 tests should pass. Combined with the real degradation shown above (FGSM and PGD both
reducing similarity below the clean baseline, PGD typically more than FGSM, both visually
near-imperceptible), Module 4 is complete. Continue to
**Module 5 — Adversarial Detection Engine** — this is where TRACER actually detects what you
just generated.